In [2]:
from pybaseball import statcast, cache
import pandas as pd

from src.data import pull_statcast
from src.features import encode_pitches, add_situation_features, add_lag_features, add_arsenal_features
from src.arsenal import compute_arsenal
import config

In [3]:
statcast_df = pull_statcast(config.SEASON, override_cache_path="../data/statcast_2025.parquet")
statcast_df.head()

Loading cached data from ../data/statcast_2025.parquet


,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,...,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches
4440,FF,2025-09-28,91.3,1.93,6.21,"Aldegheri, Sam",805904,691951,None,ball,...,<NA>,1.31,0.35,0.35,41.3,<NA>,<NA>,<NA>,<NA>,<NA>
4188,FF,2025-09-28,93.1,2.08,6.12,"Aldegheri, Sam",805904,691951,None,called_strike,...,<NA>,1.19,0.72,0.72,37.8,<NA>,<NA>,<NA>,<NA>,<NA>
4135,FF,2025-09-28,91.8,2.06,6.26,"Aldegheri, Sam",805904,691951,None,ball,...,<NA>,1.3,0.41,0.41,37.8,<NA>,<NA>,<NA>,<NA>,<NA>
3894,FF,2025-09-28,92.5,2.05,6.2,"Aldegheri, Sam",805904,691951,None,ball,...,<NA>,1.17,0.36,0.36,39.2,<NA>,<NA>,<NA>,<NA>,<NA>
3823,FF,2025-09-28,92.7,1.99,6.21,"Aldegheri, Sam",805904,691951,None,called_strike,...,<NA>,1.22,0.54,0.54,39.7,<NA>,<NA>,<NA>,<NA>,<NA>


In [4]:
statcast_df["pitch_type"].value_counts()

pitch_type
FF    237316
SI    115375
SL    107554
CH     76338
ST     56510
FC     55902
CU     49854
FS     24674
KC     13251
SV      3738
EP       987
FA       901
FO       828
CS       409
KN       140
PO        55
UN        14
SC         7
Name: count, dtype: int64

- Take out disallowed pitches (eephus, pitchout, etc.)
- Collapse slow curve with regular curve
- Map to indices

In [5]:
df = encode_pitches(statcast_df)

df["pitch_type_idx"].value_counts()

pitch_type_idx
0     237316
1     115375
2     107554
3      76338
4      56510
5      55902
6      50263
7      24674
8      13251
9       3738
10       828
11       140
Name: count, dtype: int64

- on_1b, on_2b, on_3b -> single 3 bit runners_on value
- Create "pitch_num_in_ab"
- Create "is_first_pitch" as an extra boolean
- One-hotify batter stance and pitcher throwing side (i.e. "stand_L", "throws_R")

In [6]:
df = add_situation_features(df)

df["runners_on"].value_counts()

runners_on
0    425090
1    139329
2     58177
3     49876
5     21177
4     17161
7     16539
6     14540
Name: count, dtype: int64

Add the following lag features:
- pitch characteristics: speed, x, y, spin, and movement
- whether the batter whiffed
- whether the batter fouled the pitch off
Sentinel value (-99) is used for non-existent past pitches

Encoding pitch characteristics shows that certain categories are related (fastballs, offspeed, breaking), but there isn't a linear order to pitch types as would exist in "pitch_type_idx".

In [7]:
df = add_lag_features(df)
df.head()

,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,...,prev_whiff_2,prev_foul_2,prev_speed_3,prev_plate_x_3,prev_plate_z_3,prev_pfx_x_3,prev_pfx_z_3,prev_spin_3,prev_whiff_3,prev_foul_3
4440,FF,2025-09-28,91.3,1.93,6.21,"Aldegheri, Sam",805904,691951,None,ball,...,0,0,-1.0,-99.000000,-99.000000,-99.00,-99.00,-1.0,0,0
4188,FF,2025-09-28,93.1,2.08,6.12,"Aldegheri, Sam",805904,691951,None,called_strike,...,0,0,-1.0,-99.000000,-99.000000,-99.00,-99.00,-1.0,0,0
4135,FF,2025-09-28,91.8,2.06,6.26,"Aldegheri, Sam",805904,691951,None,ball,...,0,0,-1.0,-99.000000,-99.000000,-99.00,-99.00,-1.0,0,0
3894,FF,2025-09-28,92.5,2.05,6.20,"Aldegheri, Sam",805904,691951,None,ball,...,0,0,91.3,-0.812232,1.264530,0.35,1.42,2373.0,0,0
3823,FF,2025-09-28,92.7,1.99,6.21,"Aldegheri, Sam",805904,691951,None,called_strike,...,0,0,93.1,-0.302552,1.806829,0.72,1.44,2358.0,0,0


Create a new dataframe with each pitcher's arsenal (how much they use each pitch type). Then add that as a feature on each pitch.

In [9]:
arsenal = compute_arsenal(df)
arsenal.head()

pitch_type,CH,CU,FC,FF,FO,FS,KC,KN,SI,SL,ST,SV
pitcher,,,,,,,,,,,,
434378,0.085450,0.146651,0.000000,0.454965,0.0,0.0,0.0,0.0,0.000000,0.233641,0.079292,0.0
445276,0.000000,0.000000,0.814410,0.000000,0.0,0.0,0.0,0.0,0.091703,0.060044,0.033843,0.0
445926,0.182292,0.052083,0.375000,0.000000,0.0,0.0,0.0,0.0,0.348958,0.041667,0.000000,0.0
448179,0.000000,0.347059,0.158824,0.370588,0.0,0.0,0.0,0.0,0.064706,0.000000,0.058824,0.0
450203,0.099294,0.383046,0.091052,0.275510,0.0,0.0,0.0,0.0,0.151099,0.000000,0.000000,0.0


In [10]:
df = add_arsenal_features(df, arsenal)

df.head()

,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,...,FC_pct,FF_pct,FO_pct,FS_pct,KC_pct,KN_pct,SI_pct,SL_pct,ST_pct,SV_pct
4440,FF,2025-09-28,91.3,1.93,6.21,"Aldegheri, Sam",805904,691951,None,ball,...,0.0,0.471947,0.0,0.0,0.0,0.0,0.0,0.174917,0.0,0.0
4188,FF,2025-09-28,93.1,2.08,6.12,"Aldegheri, Sam",805904,691951,None,called_strike,...,0.0,0.471947,0.0,0.0,0.0,0.0,0.0,0.174917,0.0,0.0
4135,FF,2025-09-28,91.8,2.06,6.26,"Aldegheri, Sam",805904,691951,None,ball,...,0.0,0.471947,0.0,0.0,0.0,0.0,0.0,0.174917,0.0,0.0
3894,FF,2025-09-28,92.5,2.05,6.20,"Aldegheri, Sam",805904,691951,None,ball,...,0.0,0.471947,0.0,0.0,0.0,0.0,0.0,0.174917,0.0,0.0
3823,FF,2025-09-28,92.7,1.99,6.21,"Aldegheri, Sam",805904,691951,None,called_strike,...,0.0,0.471947,0.0,0.0,0.0,0.0,0.0,0.174917,0.0,0.0
